# 21 Self-Labeled Data Check

**Purpose:** Listen to the self-labeled YouTube spans and check how they are split by dog and cut into windows for the `audio_context` task.

**Inputs:** `<data dir>/raw/self_labeled/audio/labels.csv` and its videos, the prepared manifest.

**Outputs:** Printed checks and audio players only; no files are written. Outputs are cleared in the saved notebook (the audio players made it about 41 MB).

**Splits used:** train and val rows are checked; test row counts are printed (test: counted, not scored).

In [ ]:
import pandas as pd
from IPython.display import Audio, display

from woofdata.path import data_path, load_data_dir_from_env_file
from woofdata.prepare.sources import self_labeled as sl
from woofdata.tasks import task_rows
from woofdecoder.preprocess.media import TARGET_SAMPLE_RATE, cut_audio, load_audio_16k

pd.set_option("display.width", 200)

In [ ]:
load_data_dir_from_env_file()

RAW_DIR = data_path("raw/self_labeled")
COLLECTOR_DIR = data_path("raw/self_labeled/audio")

## Listen to the labelled spans

Play the labelled span of every video in `labels.csv`.

In [ ]:
labels = pd.read_csv(COLLECTOR_DIR / "labels.csv")

In [ ]:
for row in labels.itertuples():
    video_path = COLLECTOR_DIR / "videos" / f"{row.video_id}.mp4"

    audio = load_audio_16k(video_path, verbose=False)
    if audio is None:
        print(f"{row.video_id}: No audio track")
        continue

    span = cut_audio(audio, row.start_s, row.end_s)

    print(
        f"{row.video_id} {row.dog_id} {row.start_s}-{row.end_s}s ({len(span) / TARGET_SAMPLE_RATE:.1f}s)"
    )
    display(Audio(span, rate=TARGET_SAMPLE_RATE))

## Spans, splits and windows

Read the collection, check the spans, split by dog and show how spans are cut into windows.

In [ ]:
# Only the "audio" collection: fusion/ has videos but no labels.csv yet.
spans = sl.read_collection(RAW_DIR, "audio")
sl.check_spans(RAW_DIR, spans)
split_by_dog = sl.assign_splits(spans)

print(len(spans), "spans |", sorted(split_by_dog.items()))
print("windows for a 25s span:", sl.window_bounds(25.0))
print("windows for a 10.4s span:", sl.window_bounds(10.4))

## Rows per split

Count `audio_context` rows per split, source and category.

In [ ]:
for split in ("train", "val", "test"):
    rows = task_rows("audio_context", split)
    print(split, len(rows))
    print(rows.groupby(["source", "mapped_category"]).size().to_string(), "\n")

## Fusion clips stay out of `audio_context`

Check that no fusion video is in the audio task and that no dog is in both train and val.

In [ ]:
train = task_rows("audio_context", "train")
val = task_rows("audio_context", "val")
mine_train = train[train["source"] == "self_labeled"]
mine_val = val[val["source"] == "self_labeled"]

in_fusion = (
    train["clip_id"].str.contains("/fusion/").sum() + val["clip_id"].str.contains("/fusion/").sum()
)

print("fusion rows in audio_context:", in_fusion)
print("dogs in both train and val:", set(mine_train["dog_id"]) & set(mine_val["dog_id"]))

## Results and decisions

- Act_Env totals 12,980 (the 12,480 train + 500 validation clips), across 4 categories.
- distressed_lonely has 62 training windows and 13 val windows.
- Test is PlayPanT only.
- distressed_lonely is 62 of 13,042 rows (0.5%), from about 16 videos.